# Phân tích kết quả RAGAS (10 Câu HotpotQA)
Notebook này giúp bạn phân tích sâu vào kết quả đánh giá để tìm ra điểm nghẽn của hệ thống RAG.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', None)

# 1. Đọc dữ liệu
df = pd.read_csv('ragbench_hotpotqa_full_evaluate.csv', encoding='utf-8-sig')
print(f"Tổng số mẫu: {len(df)}")

### 1. Tổng quan 4 Metrics của RAGAS

In [ ]:
# Tính điểm trung bình
metrics = ['faithfulness', 'answer_relevance', 'context_recall', 'context_precision']
means = df[metrics].mean()

# Vẽ biểu đồ Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(x=means.index, y=means.values, palette='viridis')
plt.title('Điểm trung bình RAGAS Metrics')
plt.ylim(0, 1.1)
for i, v in enumerate(means.values):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
plt.show()

### 2. Phân tích lỗi Retrieval (Tìm kiếm kém)
Lọc ra các câu có `context_recall == 0.0`. Đây là những câu hệ thống **không tìm được đoạn văn chứa thông tin trả lời**.

In [ ]:
bad_recall = df[df['context_recall'] == 0.0]
print(f"Có {len(bad_recall)} câu bị lỗi Retrieval (Tìm thiếu thông tin).")

# Hiển thị chi tiết 1 câu lỗi để phân tích
if not bad_recall.empty:
    sample = bad_recall.iloc[0]
    print("\n" + "="*60)
    print("CÂU HỎI:\n", sample['question'])
    print("\nĐÁP ÁN CHUẨN:\n", sample['ground_truth'])
    print("\nĐOẠN VĂN TÌM ĐƯỢC (QDRANT):\n", sample['documents'][:1000] + "...")
    print("="*60)

### 3. Phân tích lỗi Generation (AI Bịa Đặt / Hallucination)
Lọc ra các câu có `faithfulness == 0.0`. Đây là những câu AI **trả lời thông tin không hề có trong tài liệu**.

In [ ]:
bad_faith = df[df['faithfulness'] == 0.0]
print(f"Có {len(bad_faith)} câu AI bịa đặt (Hallucination).")

# Hiển thị chi tiết
if not bad_faith.empty:
    sample = bad_faith.iloc[0]
    print("\n" + "="*60)
    print("CÂU HỎI:\n", sample['question'])
    print("\nĐOẠN VĂN TÌM ĐƯỢC:\n", sample['documents'][:1000] + "...")
    print("\nCÂU TRẢ LỜI CỦA AI (Chứa thông tin bịa đặt):\n", sample['response_AI'])
    print("="*60)